# 딥러닝 챌린지 2026 — 베이스라인 (해설판)

Qwen2.5-3B-Instruct로 수학 문제를 풀고, 같은 문제를 여러 번 풀어서 **다수결**로 답을 정합니다.

## 실행 전 체크리스트
1. 우측 Settings → **Accelerator = GPU T4 x2**
2. 우측 Settings → **Internet = On** (전화번호 인증 필요)
3. 첫 셀을 돌려서 `/kaggle/input/` 아래 파일이 보이는지 확인. 안 보이면 **+ Add Input**으로 대회 데이터 추가

## 진행 순서
`MODE`만 바꿔가며 세 번 돌립니다.

| 순서 | MODE | 무엇을 하나 | 걸리는 시간 |
|---|---|---|---|
| 1 | `"smoke"` | 20문제만. **파이프라인이 끝까지 도는지 확인** | 5분 |
| 2 | `"valid"` | train 300문제로 **내 정확도 측정** | 20~40분 |
| 3 | `"leaderboard"` | 831문제 전체 → `submission.csv` | 1~3시간 |

> ⚠️ 최종 `test.csv`는 **8/31 00:00 공개, 당일 23:59 마감**입니다.
> 8/31 전에 리허설을 끝내두고, 그 주 Kaggle GPU 할당량을 남겨두세요.


---
## 1. 설정

### 📌 이 셀이 하는 일
실험 조건을 한곳에 모아둡니다. 아래 셀들은 전부 여기 값을 읽어서 동작하므로, **바꿀 게 있으면 여기만 고치면 됩니다.**

### 🤔 왜 이렇게 했나
- **`N_SAMPLES`**: 같은 문제를 몇 번 풀게 할지. 이게 이 대회의 핵심 손잡이입니다. 늘리면 정확도가 오르고 시간이 늘어납니다.
- **`TEMP` (temperature)**: 모델이 다음 단어를 고를 때의 무작위성. `0`이면 항상 똑같은 답이 나와서 여러 번 풀리는 의미가 없고, 너무 높으면 헛소리를 합니다. `0.7~0.8`이 다수결용 표준값입니다.
- **`TP_SIZE=1`**: GPU 한 장만 씁니다. 두 장(`2`)이 더 빠르지만 통신 설정에서 깨지는 경우가 있어, 처음엔 안전하게 1로 시작합니다.

### 🔧 바꿔볼 것
smoke 성공 → `N_SAMPLES`를 8, 16, 32로 올려가며 `valid` 정확도가 얼마나 오르는지 기록해 보세요. 보통 어느 지점부터 거의 안 오릅니다. **그 지점이 최적값**입니다.

In [ ]:
# ── 여기만 바꿔가며 실험하세요 ────────────────────────────
MODE       = "smoke"    # "smoke" | "valid" | "leaderboard"
N_SAMPLES  = 4          # 문제당 생성 횟수. smoke=4, valid=8, leaderboard=16~32
TEMP       = 0.8        # 무작위성. N_SAMPLES=1이면 자동으로 0.0(항상 같은 답)
MAX_TOKENS = 1024       # 풀이 하나의 최대 길이(토큰)
VALID_N    = 300        # MODE="valid"에서 train에서 뽑을 검증 문제 수
TP_SIZE    = 1          # 1=GPU 한 장(안전), 2=두 장 병렬(빠름)
SEED       = 42         # 고정하면 같은 조건에서 같은 결과가 재현됨
MODEL_ID   = "Qwen/Qwen2.5-3B-Instruct"   # 규칙상 고정. 절대 변경 금지
# ──────────────────────────────────────────────────────────
print(f"MODE={MODE} | N_SAMPLES={N_SAMPLES} | TEMP={TEMP} | TP_SIZE={TP_SIZE}")

---
## 2. vLLM 설치

### 📌 이 셀이 하는 일
`vLLM`이라는 추론 엔진을 설치합니다.

### 🤔 왜 vLLM인가 (그냥 transformers 쓰면 안 되나?)
됩니다. 근데 **수십 배 느립니다.**

우리는 831문제 × 16샘플 = **13,296번**을 생성해야 합니다. transformers의 기본 `generate()`는 한 번에 하나씩 처리해서 며칠이 걸립니다. vLLM은 **PagedAttention**이라는 기법으로 수백 개 요청을 GPU에 동시에 밀어넣고, 끝난 자리에 바로 다음 요청을 채웁니다. 8/31 24시간 안에 끝내려면 이게 필수입니다.

### ⚠️ 자주 나는 에러
설치하면 torch 버전이 바뀝니다. `import vllm`에서 CUDA 관련 에러가 나면 정상입니다.
→ 상단 **Run → Restart Session** 하고, **이 셀은 건너뛴 채 3번부터** 다시 실행하세요. (설치는 세션에 남아 있습니다)

In [ ]:
!pip install -q -U vllm 2>&1 | tail -3

import vllm, torch
print("vllm :", vllm.__version__)
print("torch:", torch.__version__)
print("GPU  :", torch.cuda.get_device_name(0), f"x{torch.cuda.device_count()}")

---
## 3. 데이터 로드 + 오류 문항 제거

### 📌 이 셀이 하는 일
`/kaggle/input/` 아래에서 파일 3개를 찾아 읽고, **train에서 오류 문항 627개를 뺍니다.**

| 파일 | 용도 |
|---|---|
| `..._train.csv` | 17,000문제 (id, question, **answer**) → 학습·검증용 |
| `..._leaderboard_filtered.csv` | 831문제 (id, question) → 실시간 순위 제출용 |
| `train_filtered_ids.csv` | **제거 대상 627개 목록** |

### 🤔 왜 오류 문항을 빼나
주최 측이 8/3 공지로 "train에 답이 틀렸거나 문제가 깨진 항목이 있다"며 목록을 줬습니다. 예를 들어 `train-000017`은 문제 본문이 이미지 링크 하나뿐입니다. **이걸 안 빼고 검증하면 절대 못 맞히는 문제 때문에 내 실력을 실제보다 낮게 측정**하게 됩니다.

`train_filtered_ids.csv`에 `answer`, `question` 컬럼도 들어 있어서 쓸 수 있는 데이터처럼 보이지만, **`id` 컬럼만 쓰세요.**

### ⚠️ 여기 함정이 있었습니다
`"train"`으로 파일을 찾으면 `train_filtered_ids.csv`도 걸립니다. 잘못 잡히면 **오류 문항 627개로 학습하는 정반대 상황**이 됩니다. 그래서 배제어와 `assert`를 걸어뒀습니다.

In [ ]:
import glob, os, pandas as pd

def find_csv(must_have, must_not=()):
    """/kaggle/input 아래 csv 중 키워드를 모두 포함하고 금지어는 없는 첫 파일."""
    for p in sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True)):
        b = os.path.basename(p).lower()
        if all(k in b for k in must_have) and not any(k in b for k in must_not):
            return p
    return None

# "train"은 train_filtered_ids.csv 에도 들어가므로 반드시 배제어를 건다
TRAIN_PATH = find_csv(["train"], must_not=["filtered", "ids", "leaderboard", "test"])
BAD_PATH   = find_csv(["filtered", "ids"])
LB_PATH    = find_csv(["leaderboard", "filtered"]) or find_csv(["leaderboard"])

print("train      :", TRAIN_PATH)
print("bad ids    :", BAD_PATH)
print("leaderboard:", LB_PATH)
assert TRAIN_PATH and LB_PATH, "경로 탐색 실패. 우측 Input 패널에서 실제 경로를 보고 직접 지정하세요."
assert TRAIN_PATH != BAD_PATH, "train 경로가 오류목록 파일로 잡혔습니다!"

train = pd.read_csv(TRAIN_PATH)
lb    = pd.read_csv(LB_PATH)
before = len(train)

if BAD_PATH:
    bad_ids = set(pd.read_csv(BAD_PATH)["id"])
    train = train[~train["id"].isin(bad_ids)].reset_index(drop=True)
    removed = before - len(train)
    print(f"\n오류 문항 제거: {before} -> {len(train)}  (제거 {removed}개)")
    assert removed > 0, "제거된 행이 0개입니다. id 형식을 확인하세요."
    assert train["id"].isin(bad_ids).sum() == 0, "필터링 후에도 오류 id가 남아 있습니다."
else:
    print("\n[경고] train_filtered_ids.csv 를 못 찾아 필터링을 건너뜁니다.")

print("train", train.shape, "| leaderboard", lb.shape)
train.head(3)

---
## 4. 답 추출기 (파싱)

### 📌 이 셀이 하는 일
모델이 뱉은 긴 풀이 텍스트에서 **최종 정수 하나만** 뽑아냅니다.

```
"...따라서 총 개수는 \boxed{132}개입니다."   →   132
```

### 🤔 왜 이게 중요한가
제출 파일의 `answer` 컬럼에는 **정수만** 들어가야 합니다. 모델이 정답을 맞혀놓고도 여기서 못 뽑아내면 그냥 오답 처리입니다. 이 대회에서 순위가 갈리는 가장 흔한 이유가 모델 성능이 아니라 **파싱 실패**입니다.

### 🤔 코드의 핵심 판단 3가지

**① `float()`을 절대 안 씁니다.**
train의 최대 답이 `3,431,577,212,128,939`(약 3.4경)입니다. float으로 변환하면 정밀도가 깨져서 다른 숫자가 됩니다. 그래서 문자열 → `int()` 경로만 씁니다.

**② 중괄호를 세면서 짝을 맞춥니다.**
`\boxed{\frac{100}{4}}` 처럼 중괄호가 겹칩니다. 단순 정규식으로는 `\frac{100` 에서 끊겨버립니다. `depth` 변수로 여는 괄호/닫는 괄호를 세는 이유입니다.

**③ `\boxed`가 있는데 정수가 아니면 `None`을 반환합니다.**
`\boxed{\frac{7}{2}}` 처럼 정수가 아니면, 텍스트에서 아무 숫자나 주워오는 대신 **"이 샘플은 무효"** 처리합니다. 무효 샘플은 다수결에서 빠집니다. **틀린 답을 투표에 넣는 것보다 기권시키는 게 낫기 때문**입니다.

### ✅ 자체 검증
아래에서 큰 정수, 음수, `\dfrac`, LaTeX 천단위 구분자, 단위 붙은 답 등을 테스트합니다. `FAILURES: 0`이 나와야 진행합니다.

In [ ]:
import re
from collections import Counter

def extract_boxed(text):
    """Return the raw content inside the LAST \\boxed{...}, brace-balanced."""
    idx = text.rfind('\\boxed')
    if idx == -1:
        return None
    i = idx + len('\\boxed')
    while i < len(text) and text[i] == ' ':
        i += 1
    if i >= len(text):
        return None
    if text[i] != '{':                       # bare form: \boxed 15
        m = re.match(r'-?[\d,]+', text[i:])
        return m.group(0) if m else None
    depth, start = 0, i + 1
    while i < len(text):
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0:
                return text[start:i]
        i += 1
    return None

def to_int(s):
    """LaTeX/text -> python int, or None. Never uses float(), so huge ints survive."""
    if s is None:
        return None
    s = str(s).strip()
    s = s.replace('{,}', '').replace('{\\,}', '')          # LaTeX thousands separator
    s = re.sub(r'\\(?:text|mathrm|mbox|textbf|textrm)\s*\{([^{}]*)\}', r'\1', s)
    for junk in ['\\!', '\\,', '\\;', '\\:', '\\ ', '\\left', '\\right',
                 '\\$', '$', '%', '~', '^\\circ', '\\%']:
        s = s.replace(junk, '')
    s = s.replace(',', '').replace(' ', '').strip()
    s = re.sub(r'[a-zA-Z]+$', '', s)                       # trailing unit: 42cm -> 42
    while len(s) > 1 and s[0] == '(' and s[-1] == ')':     # (\frac{100}{4}) -> \frac{100}{4}
        s = s[1:-1].strip()
    s = s.rstrip('.')
    if not s:
        return None
    m = re.fullmatch(r'\\[dt]?frac\{([-+]?\d+)\}\{([-+]?\d+)\}', s)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        return a // b if b != 0 and a % b == 0 else None
    m = re.fullmatch(r'([-+]?\d+)/([-+]?\d+)', s)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        return a // b if b != 0 and a % b == 0 else None
    m = re.fullmatch(r'([-+]?\d+)(?:\\times|\\cdot)10\^\{?(\d+)\}?', s)
    if m:
        return int(m.group(1)) * 10 ** int(m.group(2))
    if re.fullmatch(r'[-+]?\d+', s):
        return int(s)
    m = re.fullmatch(r'([-+]?\d+)\.0*', s)
    if m:
        return int(m.group(1))
    return None

def last_int(text):
    for c in reversed(re.findall(r'-?\d[\d,]*', text)):
        v = to_int(c)
        if v is not None:
            return v
    return None

def parse_answer(text):
    """None means 'this sample produced no usable integer' -> dropped from voting."""
    raw = extract_boxed(text)
    if raw is not None:
        return to_int(raw)          # boxed present but unparseable -> None, do NOT guess
    m = re.findall(r'(?:answer|Answer|ANSWER)\s*(?:is|:|=)+\s*\$?(-?[\d,]+)', text)
    if m:
        v = to_int(m[-1])
        if v is not None:
            return v
    return last_int(text)

def majority_vote(values, fallback=0):
    vals = [v for v in values if v is not None]
    if not vals:
        return fallback
    return Counter(vals).most_common(1)[0][0]


# ── 자체 검증: FAILURES가 0이 아니면 고치고 진행하세요 ──
_cases = [
    (r"총합은 \boxed{132}.",                      132),
    (r"\boxed{-2,025,078}",                       -2025078),   # 음수 + 천단위 쉼표
    (r"\boxed{3{,}431{,}577{,}212{,}128{,}939}",  3431577212128939),  # 초대형 정수
    (r"Final: \boxed{\dfrac{650}{5}}",            130),        # 나누어떨어지는 분수
    (r"\boxed{42 \text{ cm}}",                    42),         # 단위가 붙은 답
    (r"\boxed{\left(\frac{100}{4}\right)}",       25),         # 중첩 괄호
    (r"\boxed{\frac{7}{2}}",                      None),       # 정수 아님 -> 무효 처리
    (r"no box, ends with 12 cats",                12),         # boxed 없을 때 폴백
]
_bad = sum(parse_answer(t) != w for t, w in _cases)
for t, w in _cases:
    got = parse_answer(t)
    print(("OK  " if got == w else "FAIL"), repr(t[:42]).ljust(46), "->", got)
print("\nFAILURES:", _bad, "/", len(_cases))

---
## 5. 프롬프트 구성

### 📌 이 셀이 하는 일
문제 텍스트를 **모델이 알아듣는 대화 형식**으로 감쌉니다.

### 🤔 chat template이 뭔가
Qwen 같은 Instruct 모델은 특정 형식으로 학습됐습니다. 대충 이런 모양입니다.

```
<|im_start|>system
너는 수학을 잘 푸는 조수다<|im_end|>
<|im_start|>user
2+2는?<|im_end|>
<|im_start|>assistant
```

문제 텍스트를 **그냥 날것으로 넣으면 성능이 크게 떨어집니다.** `apply_chat_template()`이 이 형식을 자동으로 붙여줍니다. 직접 문자열로 짜지 마세요, 모델마다 다릅니다.

### 🤔 시스템 프롬프트를 이렇게 쓴 이유
- `"step by step"` → 바로 답을 뱉는 대신 풀이 과정을 쓰게 합니다. 수학에서 정확도가 크게 오릅니다 (Chain-of-Thought)
- `"ALWAYS a single integer"` → 답이 정수라는 걸 미리 알려줍니다. 분수나 소수로 답하는 걸 줄입니다
- `"inside \boxed{}"` → 4번 셀의 파서가 찾는 형식으로 답을 내게 합니다

### 🔧 바꿔볼 것
`valid` 정확도가 안 나오면 **모델을 바꾸기 전에 이 문장부터 손보세요.** 프롬프트 한 줄이 몇 시간짜리 학습보다 효과가 클 때가 많습니다.

In [ ]:
from transformers import AutoTokenizer

SYSTEM = ("You are an expert competition mathematician. Solve the problem step by step, "
          "concisely. The final answer is ALWAYS a single integer. "
          "End your response with the final integer inside \\boxed{}.")

if MODE == "smoke":
    work, gold = lb.head(20).copy(), None
elif MODE == "valid":
    work = train.sample(VALID_N, random_state=SEED).reset_index(drop=True)
    gold = work["answer"].tolist()          # 정답을 알고 있으므로 채점 가능
elif MODE == "leaderboard":
    work, gold = lb.copy(), None
else:
    raise ValueError(f"모르는 MODE: {MODE}")

tok = AutoTokenizer.from_pretrained(MODEL_ID)
prompts = [
    tok.apply_chat_template(
        [{"role": "system", "content": SYSTEM},
         {"role": "user",   "content": q}],
        tokenize=False, add_generation_prompt=True)
    for q in work["question"]
]

print(f"[{MODE}] {len(prompts)}문제 x {N_SAMPLES}샘플 = 총 {len(prompts)*N_SAMPLES}번 생성\n")
print("--- 실제 모델에 들어가는 문자열 (첫 문제) ---")
print(prompts[0][:600])

---
## 6. 모델 로드 + 생성

### 📌 이 셀이 하는 일
GPU에 모델을 올리고, 모든 문제를 `N_SAMPLES`번씩 풀게 합니다. **가장 오래 걸리는 셀입니다.**

### 🤔 옵션 하나하나 (면접에서 물어볼 만한 것들)

**`dtype="half"` — 왜 fp16인가**
`half`는 fp16(16비트 부동소수점)입니다. 보통 요즘은 bf16을 쓰는데, **T4는 bf16을 지원하지 않습니다.** T4는 Turing 아키텍처(2018)이고 bf16은 Ampere(2020)부터 들어갔습니다. 여기서 `bfloat16`을 쓰면 에러가 납니다.
> fp16과 bf16은 둘 다 16비트지만 비트를 나누는 방식이 다릅니다. bf16은 표현 **범위**가 넓어(오버플로에 강함) 학습에 유리하고, fp16은 **정밀도**가 높습니다. 추론만 할 거면 fp16으로 충분합니다.

**`gpu_memory_utilization=0.90` — GPU 메모리의 90%를 미리 잡습니다**
남은 공간은 **KV cache**로 씁니다. KV cache는 이미 생성한 토큰의 계산 결과를 저장해두는 공간인데, 이게 클수록 **동시에 처리할 수 있는 요청 수가 늘어나** 전체가 빨라집니다. 3B 모델은 fp16으로 약 6GB, T4는 16GB이므로 10GB가 KV cache로 갑니다.

**`max_model_len=4096` — 입력 + 출력 총 길이 상한**
문제가 길어야 400토큰 정도라 `1024(출력) + 여유`로 충분합니다. 이걸 무작정 키우면 KV cache가 커져서 오히려 느려집니다.

**`seed=SEED` — 재현성**
같은 조건에서 같은 결과가 나옵니다. "어제는 되던 게 오늘 안 되는" 상황을 막아줍니다.

### ⏱️ 시간이 오래 걸리면
`MAX_TOKENS`를 512로 줄이거나, `N_SAMPLES`를 낮추세요. 8번 셀에서 예산을 계산해줍니다.

In [ ]:
import time
from vllm import LLM, SamplingParams

llm = LLM(
    model=MODEL_ID,
    dtype="half",                    # T4는 bf16 불가 -> fp16
    max_model_len=4096,              # 입력+출력 총 토큰 상한
    gpu_memory_utilization=0.90,     # 나머지는 KV cache로 사용
    tensor_parallel_size=TP_SIZE,
    seed=SEED,
    trust_remote_code=True,
)

sp = SamplingParams(
    n=N_SAMPLES,                                  # 문제당 몇 개 뽑을지
    temperature=0.0 if N_SAMPLES == 1 else TEMP,  # 1개면 무작위성 끄기
    top_p=0.95,                                   # 확률 상위 95% 안에서만 고름
    max_tokens=MAX_TOKENS,
    seed=SEED,
)

t0 = time.time()
outs = llm.generate(prompts, sp)     # ← 여기서 실제 생성이 일어남
elapsed = time.time() - t0

print(f"\n생성 완료: {elapsed/60:.1f}분 ({elapsed/len(prompts):.2f}초/문제)")
print("\n--- 첫 문제의 첫 번째 풀이 ---")
print(outs[0].outputs[0].text[:800])

---
## 7. 다수결 + 제출 파일 저장

### 📌 이 셀이 하는 일
문제마다 나온 `N_SAMPLES`개의 답 중 **가장 많이 나온 값**을 최종 답으로 정하고 `submission.csv`를 씁니다.

### 🤔 다수결(Self-Consistency)이 왜 통하는가
이게 이 대회 전략의 핵심입니다.

> **틀리는 방법은 여러 가지지만, 맞는 방법은 하나뿐입니다.**

모델이 같은 문제를 16번 풀면, 계산 실수는 매번 다른 곳에서 다른 값으로 납니다. 그래서 **오답은 흩어지고 정답은 한 곳에 뭉칩니다.**

```
16개 답: [42, 42, 37, 42, 51, 42, 42, 8, 42, ...]
          → 42가 압도적 → 42 선택
```

학습을 전혀 안 하고도 정확도를 크게 올릴 수 있는 방법이고, 규칙에도 명시적으로 허용돼 있습니다.

### 🤔 파싱 실패 시 왜 `0`인가
빈 값을 넣으면 무조건 오답이고 제출 형식도 깨집니다. 반면 train에서 답이 `0`인 문제가 210개 있으므로, **찍더라도 0이 가장 나은 선택**입니다.

### 📊 파싱 실패율을 꼭 보세요
10%를 넘으면 모델이 아니라 **프롬프트나 파서가 문제**입니다. 5번 셀의 `SYSTEM`을 손보는 게 먼저입니다.

In [ ]:
import numpy as np
from collections import Counter

preds, n_fail = [], 0
for o in outs:
    vals = [parse_answer(c.text) for c in o.outputs]   # N_SAMPLES개의 답 후보
    n_fail += sum(v is None for v in vals)
    preds.append(majority_vote(vals, fallback=0))       # 최빈값 선택

total = len(outs) * N_SAMPLES
print(f"샘플 파싱 실패율: {n_fail}/{total} = {n_fail/total:.1%}  (10% 초과면 프롬프트/파서 점검)")

sub = pd.DataFrame({"ID": work["id"], "answer": [int(p) for p in preds]})
sub["answer"] = sub["answer"].astype("int64")     # 큰 정수 보존
sub.to_csv("submission.csv", index=False)
print("\nsubmission.csv 저장 완료:", sub.shape)
print(sub.head())

if gold is not None:
    acc = np.mean([int(p) == int(g) for p, g in zip(preds, gold)])
    print(f"\n>>> 로컬 검증 정확도: {acc:.4f}  ({int(acc*len(gold))}/{len(gold)})")

    # 몇 개나 만장일치였는지 = 모델이 얼마나 확신하는지
    agree = [Counter([v for v in [parse_answer(c.text) for c in o.outputs] if v is not None])
             for o in outs]
    conf = [ (c.most_common(1)[0][1] / N_SAMPLES) if c else 0 for c in agree ]
    print(f"    평균 득표율: {np.mean(conf):.2f}  (1.0에 가까울수록 모델이 확신)")

---
## 8. 8/31 실행 시간 예산

### 📌 이 셀이 하는 일
지금 측정된 속도로 **최종 test 추론이 몇 시간 걸릴지** 계산합니다.

### 🤔 왜 이게 중요한가
8/31에 주어지는 시간은 24시간이고, Kaggle 세션은 **12시간에 강제 종료**됩니다. 게다가 중간에 크래시가 날 수도 있죠. 그래서 **6~8시간 안에 끝나는 설정**을 골라야 합니다.

`N_SAMPLES`를 무작정 키우면 정확도는 오르지만 여기서 터집니다. **정확도와 시간의 교환**이고, 그 지점을 숫자로 정하는 게 이 셀입니다.

In [ ]:
TEST_N = 1000   # 최종 test 문제 수 추정 (leaderboard가 831이므로 넉넉하게)

per_sample = elapsed / len(prompts) / N_SAMPLES     # 문제 1개 x 샘플 1개당 초
print(f"측정값: 샘플 1개당 {per_sample:.3f}초\n")
for n in [8, 16, 32, 64]:
    est = per_sample * n * TEST_N / 3600
    tag = " <-- 안전" if est < 6 else (" <-- 위험" if est > 8 else " <-- 아슬아슬")
    print(f"  N_SAMPLES={n:>3}  ->  약 {est:5.1f}시간{tag}")

print("\n※ TP_SIZE=2 (GPU 2장)로 바꾸면 대략 절반으로 줄어듭니다.")

---
# 📚 핵심 개념 정리

| 용어 | 뜻 | 왜 여기 나오나 |
|---|---|---|
| **vLLM** | 고속 LLM 추론 엔진 | 13,000번 생성을 몇 시간에 끝내려고 |
| **PagedAttention** | KV cache를 페이지 단위로 관리하는 기법 | vLLM이 빠른 이유 |
| **KV cache** | 이미 만든 토큰의 계산 결과 저장소 | 클수록 동시 처리량 증가 |
| **chat template** | 모델이 학습된 대화 형식 | 안 쓰면 성능이 크게 떨어짐 |
| **CoT (Chain-of-Thought)** | 단계별로 풀이를 쓰게 하는 것 | 수학 정확도가 크게 오름 |
| **temperature** | 다음 단어 선택의 무작위성 | 0이면 매번 같은 답 → 다수결 무의미 |
| **Self-Consistency** | 여러 번 풀고 다수결 | 학습 없이 정확도를 올리는 최대 무기 |
| **fp16 / bf16** | 16비트 실수 표현 방식 2종 | T4는 bf16 불가 → fp16 |

---

# 🎤 예상 질문 & 답변

발표나 질의응답이 있다면 이 정도는 스스로 말할 수 있어야 합니다.

**Q. 왜 Qwen2.5-3B-Instruct인가요?**
대회 규칙 4.1에서 유일한 출발점으로 지정했습니다. 다른 모델을 베이스로 쓰거나 가중치를 병합하는 것은 금지입니다.

**Q. `dtype="half"`로 한 이유는?**
Kaggle의 T4 GPU가 Turing 아키텍처라 bf16을 지원하지 않습니다. bf16을 쓰면 에러가 나서 fp16을 썼습니다.

**Q. `N_SAMPLES`를 그 값으로 정한 근거는?**
`valid` 모드로 8/16/32를 각각 측정해서 정확도 향상폭과 소요 시간을 비교했습니다. (← **실제로 측정하고 숫자를 기록해두세요.** 이 답변의 설득력은 전적으로 그 로그에서 나옵니다)

**Q. temperature를 0.8로 한 이유는?**
0에 가까우면 16개 샘플이 거의 동일해져서 다수결의 의미가 사라지고, 너무 높으면 풀이가 무너집니다. 다양성과 품질의 균형점으로 통용되는 값입니다.

**Q. 파싱 실패 시 0을 넣는 이유는?**
빈 값은 무조건 오답이고 제출 형식도 깨집니다. train에서 정답이 0인 문항이 210개 있어, 찍더라도 0의 기대값이 가장 높습니다.

**Q. `\boxed{\frac{7}{2}}` 같은 걸 왜 버리나요?**
답은 항상 정수인데 분수가 나왔다면 그 풀이는 틀린 겁니다. 텍스트에서 아무 숫자나 주워 투표에 넣으면 **정답 쪽 표를 깎는 노이즈**가 됩니다. 기권시키는 게 낫습니다.

**Q. 왜 `float()`을 안 쓰나요?**
train 최대 답이 약 3.4×10¹⁵입니다. float으로 변환하면 정밀도가 손실돼 다른 값이 됩니다.

**Q. train에서 627개를 왜 뺐나요?**
주최 측이 8/3 공지로 오류가 확인된 문항 목록을 배포했습니다. 답이 틀렸거나 문제 본문이 이미지 링크뿐인 것들이라, 포함하면 검증 정확도가 실제보다 낮게 나옵니다.

**Q. 리더보드가 있는데 왜 따로 검증셋을 만드나요?**
리더보드는 831문제뿐이고 제출 횟수도 제한됩니다. 거기 맞춰 튜닝하면 **public에만 과적합**돼서 최종 순위에서 떨어집니다. train에서 뽑은 300문제로 자유롭게 측정하는 편이 안전합니다.

---

# ✅ 오늘의 할 일

1. `MODE="smoke"` 로 끝까지 실행 → 에러 없이 `submission.csv`가 나오면 성공
2. `MODE="valid"`, `N_SAMPLES=8` → **내 기준 정확도 숫자 하나 확보**
3. 그 숫자를 적어두세요. 앞으로의 모든 개선은 이 숫자와 비교합니다
